<a href="https://colab.research.google.com/github/Zikirullo/clear-vision/blob/develop/notebooks/02_merge_datasets.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 02 · Merge datasets

Combines the raw datasets on Drive (`MyDrive/clear_vision/datasets/raw/`) and the new
sources downloaded in section 2 (`/content/raw/`) into **one file on Drive**:

```
MyDrive/clear_vision/datasets/merged.zip          YOLO dataset (ZIP_STORED)
MyDrive/clear_vision/datasets/merge_report.json   boxes per class per split, per source
MyDrive/clear_vision/datasets/review/             sample images to review
MyDrive/clear_vision/datasets/merged.done         written only after a verified upload
```

- **Why a zip:** an earlier merged *folder* didn't fit on Drive, and the background upload
  silently lost most of its files. One file can be checked after upload.
- **What the merge does:**
  - Images over `MAX_SIDE` px are resized (JPEG q90).
  - Everything is built on the Colab disk.
  - It checks free Drive space before writing, flushes Drive, and verifies the zip's
    size and CRCs.
- **Mappings:** the class list is `configs/classes.yaml`, and each source has a mapping
  file `configs/mappings/<name>.yaml`. The merge fails if a source class isn't mapped.
- **Append** (`APPEND = True`) adds `NEW_SOURCES` to the existing zip instead of
  rebuilding it. It also re-maps merged sources whose mapping file changed, for example
  `pedestrian-crossing: drop` → `crosswalk` in `rf_crosswalk.yaml`.
- **Training** notebooks load the dataset with `src/colab_data.py` `prepare_dataset()`,
  never from Drive directly (see section 4).
- No GPU is needed. Run all cells.


In [ ]:
# Settings
REPO_URL = "https://github.com/Zikirullo/clear-vision.git"
BRANCH = "develop"
DRIVE_ROOT = "/content/drive/MyDrive/clear_vision"
MERGED_ZIP = f"{DRIVE_ROOT}/datasets/merged.zip"
WORK = "/content/merged_tmp"  # local build folder (Colab disk)
SEED = 42
MAX_SIDE = 1280  # images are resized so the longer side is at most this (JPEG q90)
FORCE = False  # True rebuilds merged.zip even if it is already done

# New Roboflow sources (configs/datasets.yaml), downloaded to LOCAL_RAW in section 2.
# The full merge and the append both read LOCAL_RAW together with the raw folder on Drive.
DOWNLOAD_NEW = ["rf_sidewalk", "rf_barricade"]  # [] = skip
LOCAL_RAW = "/content/raw"

# Append mode: add sources to the existing merged.zip instead of rebuilding it.
APPEND = False
NEW_SOURCES = ["rf_sidewalk", "rf_barricade"]


## 1. Setup

In [ ]:
# 1. Setup: Drive, repo, packages
import logging
import subprocess
import sys
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

REPO_DIR = Path("/content/clear-vision")
if (REPO_DIR / ".git").exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "-q", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "-q", "-B", BRANCH, f"origin/{BRANCH}"], check=True)
else:
    subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO_URL, str(REPO_DIR)], check=True)
commit = subprocess.run(["git", "-C", str(REPO_DIR), "log", "--oneline", "-1"], capture_output=True, text=True).stdout
print(f"Repo at {REPO_DIR}: {commit.strip()}")

%pip install -q ultralytics roboflow

sys.path.insert(0, str(REPO_DIR))
sys.path.insert(0, str(REPO_DIR / "scripts"))
import download_datasets  # noqa: E402
import merge_datasets  # noqa: E402
from src.colab_data import prepare_dataset  # noqa: E402

logging.basicConfig(level=logging.INFO, format="%(message)s", force=True)

RAW = Path(DRIVE_ROOT) / "datasets" / "raw"
MERGED_ZIP = Path(MERGED_ZIP)
print(f"raw on Drive: {RAW}")
print(f"merged zip:   {MERGED_ZIP}")
print(f"Drive free:   {merge_datasets.gb(merge_datasets.free_space(MERGED_ZIP.parent))}")


## 2. Download new sources

Downloads `DOWNLOAD_NEW` (Roboflow, listed in `configs/datasets.yaml`) to `LOCAL_RAW` with the same code as notebook 01. Finished downloads (with `.done`) are skipped. Then it prints classes missing from their mapping files.

In [ ]:
# 2. Download new sources to this runtime's disk (LOCAL_RAW)
# Each Colab notebook has its own runtime, so downloads from notebook 01 are not here.
if DOWNLOAD_NEW:
    new_datasets = download_datasets.select_datasets(
        download_datasets.load_roboflow_datasets(REPO_DIR / "configs" / "datasets.yaml"), DOWNLOAD_NEW
    )
    failures = download_datasets.download_roboflow_datasets(
        new_datasets, Path(LOCAL_RAW), download_datasets.colab_secret("ROBOFLOW_API_KEY")
    )
    download_datasets.check_mappings(
        Path(LOCAL_RAW), DOWNLOAD_NEW, download_datasets.mappings_from_dir(REPO_DIR / "configs" / "mappings")
    )
    if failures:
        raise RuntimeError(f"Downloads failed, fix them before merging: {failures}")
else:
    print("DOWNLOAD_NEW is empty: nothing to download")


## 3. Merge or append

Prints the mappings, the raw sizes on Drive, the image size before/after resize, the zip size, the space check, the verification and the Drive space left. It stops with a message (and no `merged.done`) if anything doesn't fit or doesn't verify.

In [ ]:
# 3. Merge or append -> merged.zip on Drive (mappings, progress, sizes, class table)
import json

CLASSES_PATH = REPO_DIR / "configs" / "classes.yaml"
MAPPINGS_DIR = REPO_DIR / "configs" / "mappings"
raw_dirs = [Path(LOCAL_RAW), RAW] if Path(LOCAL_RAW).is_dir() else [RAW]  # local copy wins

if APPEND:
    report = merge_datasets.append(
        raw=raw_dirs,
        zip_path=MERGED_ZIP,
        new_sources=NEW_SOURCES,
        work=WORK,
        classes_path=CLASSES_PATH,
        mappings_dir=MAPPINGS_DIR,
        seed=SEED,
        max_side=MAX_SIDE,
    )
else:
    report = merge_datasets.merge(
        raw=raw_dirs,
        zip_path=MERGED_ZIP,
        work=WORK,
        classes_path=CLASSES_PATH,
        mappings_dir=MAPPINGS_DIR,
        seed=SEED,
        max_side=MAX_SIDE,
        force=FORCE,
    )
if report is None:  # nothing to do: show the saved report
    report = json.loads(MERGED_ZIP.with_name("merge_report.json").read_text())
    merge_datasets.log_summary(report)


## 4. Load and check (as training will)

In [ ]:
# 4. Load the dataset the way training will: copy merged.zip to the Colab disk, unzip,
#    compare image counts with merge_report.json, run Ultralytics' check_det_dataset.
#    Later notebooks (report, training) use exactly this line.
DATA_YAML = prepare_dataset(MERGED_ZIP)
print(f"OK: {DATA_YAML}")


## 5. Review

Decide whether `pedestrian-crossing` should map to `crosswalk` or stay dropped. To switch it, edit `configs/mappings/rf_crosswalk.yaml`, push, and run this notebook with `APPEND = True`. Only the rf_crosswalk labels are re-mapped.

In [ ]:
# 5. Review samples: pedestrian-crossing boxes (red) that are dropped for now
import matplotlib.image as mpimg
import matplotlib.pyplot as plt

for folder in sorted((MERGED_ZIP.parent / "review").glob("*")):
    files = sorted(folder.glob("*"))[:16]
    print(f"{folder.name}: {len(files)} images (red = review class, green = kept boxes)")
    if not files:
        continue
    cols = 4
    rows = (len(files) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(16, 4 * rows))
    for ax in axes.flat:
        ax.axis("off")
    for ax, path in zip(axes.flat, files):
        ax.imshow(mpimg.imread(path))
        ax.set_title(path.name[:40], fontsize=8)
    plt.tight_layout()
    plt.show()
